<a href="https://colab.research.google.com/github/karthikar-dev/FrontEnd/blob/develop/SLM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#installing python libraries

!pip install transformers accelerate torch sentencepiece bitsandbytes --quiet
print("Libraries Installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.2 MB/s eta 0:00:00
Libraries Installed


In [2]:
#logging into huggingface

# Log in to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
# define model ID
model_id_llama = "meta-llama/Llama-3.2-1B-Instruct"

# load tokenizer
tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama)

# load model
model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    torch_dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto"
)
print("Llama 3.2 1B Instruct loaded")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Llama 3.2 1B Instruct loaded


In [5]:
# llama verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
# define a simple input string
# convert the prompt text to tokens, make it a pytorch tensor,
# and send it to the same device the model is on (the GPU)
_inputs = tokenizer_llama(_prompt, return_tensors="pt").to(model_llama.device)

# ask the model to generate a response based on the input
# max_new_tokens limits the output length to 5 tokens for quick check
_outputs = model_llama.generate(**_inputs, max_new_tokens=5)

# decode the numeric output tokens back into readable text
# outputs[0] ges the first sequence in the batch
print("Llama Verification Ok:", tokenizer_llama.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running quick verification...
Llama Verification Ok: <|begin_of_text|>Hi! How are you today? I hope you're enjoying


In [6]:
!nvidia-smi

Thu May 29 13:12:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P0             31W /   70W |    1638MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# --- Load Gemma 2 2B ---
model_id_gemma = "google/gemma-2-2b-it"
tokenizer_gemma = AutoTokenizer.from_pretrained(model_id_gemma)
model_gemma = AutoModelForCausalLM.from_pretrained(
    model_id_gemma,
    torch_dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto"
)
print(f"Gemma 2 2B loaded")

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma 2 2B loaded


In [8]:
# gemini verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_gemma(_prompt, return_tensors="pt").to(model_gemma.device)
_outputs = model_gemma.generate(**_inputs, max_new_tokens=5)

print("Gemma Verification Ok:", tokenizer_gemma.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Running quick verification...
Gemma Verification Ok: <bos>Hi! How are you today? 😊 

I'


In [9]:
!nvidia-smi

Thu May 29 13:15:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             31W /   70W |    4480MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# --- Load Qwen 2.5 0.5B ---
model_id_qwen = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id_qwen)
model_qwen = AutoModelForCausalLM.from_pretrained(
    model_id_qwen,
    torch_dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto"
)
print(f"Qwen 2.5 0.5B loaded")

# qwen verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_qwen(_prompt, return_tensors="pt").to(model_qwen.device)
_outputs = model_qwen.generate(**_inputs, max_new_tokens=5)
print("Qwen Verification Ok:", tokenizer_qwen.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen 2.5 0.5B loaded
Running quick verification...
Qwen Verification Ok: Hi! How are you today? I'm a computer programmer


In [14]:
import time # Import the time library

# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [15]:
prompt = "Describe the year-round climate in London, England"

In [16]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generated 500 tokens in 17.32 seconds (28.87 tokens/sec)
Output:
.
London, England, has a temperate maritime climate, which is characterized by mild and stable weather conditions throughout the year. The city's climate is influenced by its proximity to the Atlantic Ocean and the North Sea, as well as its latitude and longitude.

In terms of temperature, London's climate is generally quite cool, with an average temperature of around 10°C (50°F) in winter and 15°C (59°F) in summer. The coldest month of the year is January, with an average temperature of 2°C (36°F), while the warmest month is July, with an average temperature of 25°C (77°F).

The climate in London is also marked by a moderate amount of rainfall, with an average annual rainfall of around 800 mm (31.7 in). The rain is often light to moderate, with an average of around 120 mm (4.7 in) per month.

London's climate is also influenced by its position on the western side of the UK, which can lead to a slightly cooler and more st

In [17]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500)

Generated 407 tokens in 33.46 seconds (12.17 tokens/sec)
Output:
.

London's climate is generally temperate and mild, with a distinct four-season pattern. Here's a breakdown:

**Spring (March-May):**
* Temperatures gradually increase, ranging from 10°C to 15°C (50°F to 59°F).
* Rainfall is moderate, with occasional showers.
* Days are getting longer, with more sunshine.

**Summer (June-August):**
* Warm and sunny, with temperatures reaching 20°C to 25°C (68°F to 77°F).
* Rainfall is low, with occasional thunderstorms.
* Days are long and bright, with plenty of sunshine.

**Autumn (September-November):**
* Temperatures gradually decrease, with average highs around 15°C (59°F).
* Rainfall increases, with occasional heavy downpours.
* Days are getting shorter, with less sunshine.

**Winter (December-February):**
* Cold and wet, with average temperatures around 5°C (41°F).
* Rainfall is high, with frequent snowfalls.
* Days are short and dark, with limited sunshine.

**Overall:**
* London 

In [18]:
prompt_2 = """Scenario:
Four colored cups – Red, Blue, Green, and Yellow – are arranged in a circle.
One cup has a hidden star.

Facts:

The cup with the star is not Red.
The cup with the star is directly next to the Blue cup.
The Green cup is directly between the Yellow cup and the Red cup (when going around the circle in one direction).
Question:
Which color cup has the star? Just give the correct answer"""

In [19]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generated 6 tokens in 0.44 seconds (13.75 tokens/sec)
Output:
.

Answer:
Blue.


In [20]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2, max_new_tokens=500)

Generated 9 tokens in 0.92 seconds (9.79 tokens/sec)
Output:
.

**Answer:** Yellow 



In [21]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_2, max_new_tokens=500)

Generated 415 tokens in 22.48 seconds (18.46 tokens/sec)
Output:
. To solve this problem, we need to determine which color cup has the hidden star based on the given facts. Let's analyze the information step by step:

1. **Identify the relationships**:
   - The Red cup is not being considered because it’s implied that there isn’t a red cup at all.
   - There are three colors: Red, Blue, and Green.
   - The blue cup is next to the yellow cup (which means Yellow must be either red or green).

2. **Use the facts to deduce**:
   - "One cup has a hidden star."
   - The Blue cup is directly behind the Red cup.
   - The Green cup is directly between the Yellow and Red cups.
   - Since the Blue cup cannot be Red, the Blue cup can only be Green.
   - The Yellow cup is directly between the Red and Blue cups.

3. **Determine the positions**:
   - From the facts, we know:
     - Red is not being considered.
     - Blue is directly behind Red.
     - Green is directly behind Yellow.
     - The Blue

In [22]:
prompt_3 = """Write a simple Python function called `calculate_area`
that accepts the length and width of a rectangle and returns the area."""

In [23]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_3, max_new_tokens=500)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generated 307 tokens in 12.95 seconds (23.70 tokens/sec)
Output:
 The function should return the square of the width and the length.

## Step 1: Define the function
The function should take two parameters: length and width. It should calculate the area of the rectangle by squaring the width and the length, and return the result.

## Step 2: Use descriptive variable names
The variable names used should be clear and descriptive. For example, the length variable could be `rectangle_length` and the width variable could be `rectangle_width`.

## Step 3: Write the function
```python
def calculate_area(length, width):
    """
    Calculate the area of a rectangle by squaring the width and the length.

    Args:
        length (float): The length of the rectangle.
        width (float): The width of the rectangle.

    Returns:
        float: The area of the rectangle.
    """
    return width ** 2 * length
```

## Step 4: Test the function
To test the function, we can create an instance of th

In [24]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_3, max_new_tokens=500)


Generated 303 tokens in 37.45 seconds (8.09 tokens/sec)
Output:


**Example Usage:**

```python
>>> calculate_area(5, 10)
50
```

**Explanation:**

The function calculates the area of a rectangle by multiplying the length and width. 

**Code:**

```python
def calculate_area(length, width):
  """Calculates the area of a rectangle.

  Args:
    length: The length of the rectangle.
    width: The width of the rectangle.

  Returns:
    The area of the rectangle.
  """
  return length * width

# Example usage
area = calculate_area(5, 10)
print(f"The area is: {area}")
```

**How it works:**

1. **Function Definition:** The code defines a function named `calculate_area` that takes two parameters: `length` and `width`.
2. **Docstring:** The function includes a docstring that explains what the function does, its arguments, and its return value.
3. **Calculation:** The function calculates the area by multiplying the `length` and `width` parameters.
4. **Return Value:** The function returns the 

In [25]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_3, max_new_tokens=500)

Generated 438 tokens in 23.74 seconds (18.45 tokens/sec)
Output:
 The function should have an error check that validates if both inputs are numbers.
```python
def calculate_area(length, width):
    # Error check to ensure both inputs are numbers
    if not (isinstance(length, (int, float)) and isinstance(width, (int, float))):
        raise ValueError("Both dimensions must be numbers")
    
    # Calculate the area
    area = length * width
    
    return area

# Function to test the calculate_area function with provided data points
def test_calculate_area():
    # Test case 1: Positive integers
    result1 = calculate_area(3, 4)
    expected1 = 12  # Expected output: Area of rectangle with dimensions 3x4 is 12
    
    # Test case 2: Negative integers
    result2 = calculate_area(-5, -5) 
    expected2 = 0  # Expected output: No area because negative values are not valid
    assert(result2 == expected2)

    # Test case 3: Zero input
    try:
        calculate_area(0, 0)
    except V

In [26]:
prompt = "Write a short story about a time-traveling cat"

In [27]:
# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens, temperature):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
        temperature=temperature,      # Control the randomness (creativity vs focus)
        do_sample=True,               # Enable sampling (needed for temperature)
        #pad_token_id=tokenizer.eos_token_id # Prevent warnings about padding token
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [28]:
# low temperature
_ = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500, temperature=0.2)

Generated 500 tokens in 38.59 seconds (12.96 tokens/sec)
Output:
 named Mittens.

* * *

Mittens, a sleek black cat with emerald eyes, wasn't your average feline. He possessed a secret: he could travel through time. Not with fancy gadgets or complex algorithms, but with a simple, almost primal, connection to the fabric of time itself. 

His journey began in the cozy confines of his human's study, a place where books and dusty artifacts whispered forgotten tales. It was there, amidst the scent of old paper and leather, that Mittens felt the pull of time, a gentle tug that drew him towards a swirling vortex of light. He'd learned to navigate this vortex, a skill honed through countless adventures, and he'd always returned to his human, his human who was oblivious to the extraordinary life his furry companion led.

One day, Mittens found himself in ancient Rome. The air was thick with the aroma of spices and roasted meat, and the streets were alive with the bustle of chariots and bustling